# 🎵 MusicGen Fine-Tuning with PEFT LoRA on Indian Instrumental Music

## Executive Summary
This notebook demonstrates fine-tuning **`facebook/musicgen-small`** using **Parameter-Efficient Fine-Tuning (PEFT)** via **Low-Rank Adaptation (LoRA)** on custom Indian classical and instrumental music data.

### Key Highlights & Technical Specs:
- **Base Model**: `facebook/musicgen-small` (Transformer decoder architecture for text-to-audio generation)
- **Fine-Tuning Method**: PEFT LoRA (Low-Rank Adaptation)
  - `r = 16`
  - `lora_alpha = 32`
  - `lora_dropout = 0.0`
  - **Target Modules**: `self_attn.q_proj`, `self_attn.v_proj`, `encoder_attn.q_proj`, `encoder_attn.v_proj`
  - **Trainable Parameters**: 3,145,728 (~3.1M trainable parameters out of ~300M total)
- **Dataset**: Indian Classical & Instrumental audio chunks (Sitar, Santoor, Bansuri Flute, Tabla, Veena, Shehnai, Sarangi, etc.) paired with detailed captions.
- **Evaluation Metrics**:
  - Validation Cross-Entropy Loss
  - CLAP (Contrastive Language-Audio Pretraining) Text-Audio Similarity Alignment Scoring

## 1. Environment Setup & Hardware Verification
We start by configuring PyTorch memory allocations and verifying GPU hardware acceleration and package dependencies.

In [ ]:
import os
import sys

# Ensure immediate unbuffered console output & optimal PyTorch memory allocation
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
import numpy as np
import scipy
import sklearn
import torch

print("NumPy Version       :", np.__version__)
print("SciPy Version       :", scipy.__version__)
print("Scikit-Learn        :", sklearn.__version__)
print("PyTorch Version     :", torch.__version__)
print("CUDA Available      :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Device Name     :", torch.cuda.get_device_name(0))
    print("GPU Memory Total    :", f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
import transformers
import accelerate
import peft
import soundfile
import librosa

print("Transformers Version:", transformers.__version__)
print("Accelerate Version  :", accelerate.__version__)
print("PEFT Version        :", peft.__version__)
print("SoundFile Version   :", soundfile.__version__)
print("Librosa Version     :", librosa.__version__)

## 2. Custom Dataset & PyTorch DataLoader
We define a custom PyTorch `Dataset` that loads preprocessed audio chunks and text captions from metadata CSVs, prepares audio waveform tensors, and tokenizes text prompts using the MusicGen `AutoProcessor`.

In [ ]:
import pandas as pd
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor
import soundfile as sf

# Set paths for metadata and audio chunks
DATASET_CSV = "data/metadata/dataset.csv"

# Load metadata
if Path(DATASET_CSV).exists():
    metadata_df = pd.read_csv(DATASET_CSV)
    print(f"Loaded metadata successfully. Total samples: {len(metadata_df)}")
    print(metadata_df[["chunk_id", "split", "genre", "musicgen_prompt"]].head())
else:
    print(f"Warning: Metadata file '{DATASET_CSV}' not found. Please verify project data paths.")

In [ ]:
class MusicDataset(Dataset):
    """
    PyTorch Dataset for MusicGen fine-tuning.
    Loads audio waveforms and corresponding text captions.
    """
    def __init__(self, metadata_csv, split="train", processor=None, sample_rate=32000):
        self.df = pd.read_csv(metadata_csv)
        if "split" in self.df.columns:
            self.df = self.df[self.df["split"] == split].reset_index(drop=True)
        self.processor = processor
        self.sample_rate = sample_rate

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = row["audio_path"]
        prompt = str(row["musicgen_prompt"])

        # Load audio waveform
        audio, sr = sf.read(audio_path, dtype="float32")

        # Convert stereo to mono if required
        if audio.ndim > 1:
            audio = np.mean(audio, axis=1)

        # Resample if sample rate differs from target (32kHz)
        if sr != self.sample_rate:
            audio = librosa.resample(audio, orig_sr=sr, target_sr=self.sample_rate)

        return {
            "audio": audio,
            "prompt": prompt,
            "chunk_id": row["chunk_id"],
        }

## 3. Model Loading & PEFT LoRA Adapter Configuration
We load the pre-trained `facebook/musicgen-small` model and configure Parameter-Efficient Fine-Tuning (LoRA). Target modules include query and value projection layers in both self-attention and cross-attention blocks.

In [ ]:
from transformers import MusicgenForConditionalGeneration
from peft import LoraConfig, TaskType, get_peft_model

MODEL_NAME = "facebook/musicgen-small"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading processor and base model: {MODEL_NAME}")
processor = AutoProcessor.from_pretrained(MODEL_NAME)
base_model = MusicgenForConditionalGeneration.from_pretrained(MODEL_NAME)

# Define LoRA Configuration
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    target_modules=[
        "self_attn.q_proj",
        "self_attn.v_proj",
        "encoder_attn.q_proj",
        "encoder_attn.v_proj",
    ],
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
)

# Wrap base model with PEFT LoRA adapter
musicgen_lora = get_peft_model(base_model, lora_config)
musicgen_lora.to(device)

print("\n" + "=" * 50)
print("LORA MODEL SUMMARY")
print("=" * 50)
musicgen_lora.print_trainable_parameters()

## 4. Fine-Tuning Training & Validation Loop
We set up the AdamW optimizer, cosine learning rate scheduler with warmup, and evaluation function to track cross-entropy validation loss.

In [ ]:
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

# Hyperparameters
NUM_EPOCHS = 15
BATCH_SIZE = 4
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01

# Initialize DataLoaders
train_dataset = MusicDataset(DATASET_CSV, split="train", processor=processor)
val_dataset = MusicDataset(DATASET_CSV, split="validation", processor=processor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

optimizer = AdamW(musicgen_lora.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
num_training_steps = NUM_EPOCHS * len(train_loader)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * num_training_steps),
    num_training_steps=num_training_steps,
)

print(f"Training initialized for {NUM_EPOCHS} epochs ({num_training_steps} total steps).")

In [ ]:
@torch.no_grad()
def evaluate(model, dataloader):
    """
    Computes average validation loss over the validation set.
    """
    model.eval()
    total_loss = 0.0
    num_batches = 0

    for batch in dataloader:
        prompts = batch["prompt"]
        audios = batch["audio"]

        # Process inputs
        inputs = processor(
            text=prompts,
            audio=audios,
            sampling_rate=32000,
            padding=True,
            return_tensors="pt",
        ).to(device)

        labels = inputs["input_values"]
        outputs = model(**inputs, labels=labels)
        loss = outputs.loss

        total_loss += loss.item()
        num_batches += 1

    model.train()
    return total_loss / max(num_batches, 1)

### Execution of Training Loop
The training loop iterates across epochs, logs loss per step, evaluates validation loss after each epoch, and checkpoints the best LoRA adapter parameters.

In [ ]:
best_val_loss = float("inf")
output_adapter_dir = "models/adapters/adapter3"

print("Starting training loop...")
for epoch in range(1, NUM_EPOCHS + 1):
    musicgen_lora.train()
    running_loss = 0.0

    for step, batch in enumerate(train_loader, start=1):
        optimizer.zero_grad()

        prompts = batch["prompt"]
        audios = batch["audio"]

        inputs = processor(
            text=prompts,
            audio=audios,
            sampling_rate=32000,
            padding=True,
            return_tensors="pt",
        ).to(device)

        labels = inputs["input_values"]
        outputs = musicgen_lora(**inputs, labels=labels)
        loss = outputs.loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(musicgen_lora.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()

    avg_train_loss = running_loss / len(train_loader)
    val_loss = evaluate(musicgen_lora, val_loader)

    print(f"Epoch [{epoch:02d}/{NUM_EPOCHS:02d}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        print(f"  --> New best validation loss: {best_val_loss:.4f}. Saving adapter to {output_adapter_dir}")
        musicgen_lora.save_pretrained(output_adapter_dir)

## 5. Audio Generation & Playback
We define a helper function `generate_audio()` to generate 30-second audio clips from text prompts using the fine-tuned model.

In [ ]:
def generate_audio(prompt, model, max_new_tokens=1500, guidance_scale=3.5, temperature=1.0, top_k=250):
    """
    Generates audio for a given prompt using the specified model.
    """
    model.eval()
    inputs = processor(text=[prompt], padding=True, return_tensors="pt").to(device)

    with torch.no_grad():
        audio_values = model.generate(
            **inputs,
            do_sample=True,
            guidance_scale=guidance_scale,
            temperature=temperature,
            top_k=top_k,
            max_new_tokens=max_new_tokens,
        )

    waveform = audio_values[0, 0].cpu().numpy()
    return waveform

In [ ]:
from IPython.display import Audio

test_prompt = (
    "Indian classical music featuring santoor, tabla, and tanpura drone. "
    "A meditative raga-style introduction gradually develops into an energetic rhythmic section."
)

print(f"Generating audio for test prompt:\n'{test_prompt}'")
generated_waveform = generate_audio(test_prompt, musicgen_lora, max_new_tokens=1500)

# Display interactive audio player in Jupyter Notebook
Audio(generated_waveform, rate=32000)

## 6. CLAP Text-Audio Alignment Evaluation
We benchmark text-audio semantic alignment using **CLAP** (`laion/clap-htsat-unfused`), comparing prompt-audio cosine similarity between the **Base MusicGen Model** and the **Fine-Tuned LoRA Adapter**.

In [ ]:
import torchaudio.transforms as T
from transformers import ClapModel, ClapProcessor

print("Loading CLAP evaluation model...")
clap_model_name = "laion/clap-htsat-unfused"
clap_processor = ClapProcessor.from_pretrained(clap_model_name)
clap_model = ClapModel.from_pretrained(clap_model_name).to(device)
clap_model.eval()

def compute_clap_similarity(texts, audio_arrays, sampling_rate=32000):
    """
    Computes cosine similarity between prompt text and generated audio embeddings using CLAP.
    """
    resample_fn = T.Resample(orig_freq=sampling_rate, new_freq=48000)
    resampled_audios = []

    for a in audio_arrays:
        t_audio = torch.tensor(a, dtype=torch.float32).unsqueeze(0)
        resampled = resample_fn(t_audio).squeeze(0).numpy()
        resampled_audios.append(resampled)

    inputs = clap_processor(
        text=texts,
        audio=resampled_audios,
        sampling_rate=48000,
        return_tensors="pt",
        padding=True,
    ).to(device)

    with torch.no_grad():
        outputs = clap_model(**inputs)
        text_features = outputs.text_embeds
        audio_features = outputs.audio_embeds

        # L2 Normalize
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        audio_features = audio_features / audio_features.norm(dim=-1, keepdim=True)

        similarity = (text_features * audio_features).sum(dim=-1)

    return similarity.cpu().numpy()

In [ ]:
# Benchmark on test set prompts
test_dataset = MusicDataset(DATASET_CSV, split="test", processor=processor)
test_prompts = test_dataset.df["musicgen_prompt"].tolist()[:10]

base_audios = []
ft_audios = []

print("Generating samples from Base and Fine-Tuned models for CLAP evaluation...")
for p in test_prompts:
    base_audios.append(generate_audio(p, base_model, max_new_tokens=500))
    ft_audios.append(generate_audio(p, musicgen_lora, max_new_tokens=500))

base_clap_scores = compute_clap_similarity(test_prompts, base_audios)
ft_clap_scores = compute_clap_similarity(test_prompts, ft_audios)

print("=" * 55)
print(f"Base Model Average CLAP Score      : {np.mean(base_clap_scores):.4f}")
print(f"Fine-Tuned Model Average CLAP Score: {np.mean(ft_clap_scores):.4f}")
print("=" * 55)

## 7. Adapter Export & Deployment Readiness
The trained PEFT LoRA adapter is exported to `models/adapters/adapter3/` containing:
- `adapter_config.json`: LoRA rank, alpha, target modules, and task type.
- `adapter_model.safetensors`: Fine-tuned adapter weight matrices.

This adapter is lightweight (~12.6 MB) and can be dynamically loaded over `facebook/musicgen-small` at runtime inside our FastAPI inference server.

In [ ]:
print("Verifying saved adapter files in models/adapters/adapter3:")
for file in Path(output_adapter_dir).glob("*"):
    print(f"  - {file.name} ({file.stat().st_size / 1e6:.2f} MB)")

print("\nFine-tuning pipeline complete! Ready for FastAPI inference deployment.")